# 03 · Recuperação e custo

Duas perguntas: a recuperação **encontra a evidência anotada**, e quanto
custa cada condição em tokens de LLM.

* `recall@k` — evidências encontradas nas top-k arestas/turnos por score;
* `recall_context` — evidências presentes no contexto realmente enviado ao
  modelo (é esta que explica o F1).


In [ ]:
from nbutils import *

# results/ por padrão; use setup(dry=True) para inspecionar um smoke run offline
ctx = setup()
ctx.conditions


## Recall@k


In [ ]:
plot_recall(ctx); show()


In [ ]:
if not ctx.recall.empty:
    display(ctx.recall.pivot_table(index='condition', columns='metric',
                                   values='value', observed=True).round(3))


## Recall do contexto, por categoria

Onde o recall do contexto cai, o F1 cai junto — é o diagnóstico mais direto
de uma falha de recuperação.


In [ ]:
sub = ctx.recall[ctx.recall.metric == 'recall_context']
if sub.empty:
    print('sem recall_context')
else:
    piv = sub.pivot_table(index='category', columns='condition',
                          values='value', observed=True)
    ax = piv.plot(kind='bar', figsize=(10, 3.6), color=colors(piv.columns), width=.82)
    ax.set_ylabel('recall do contexto enviado'); ax.set_xlabel('')
    ax.set_title('A evidência anotada chegou ao prompt?')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    plt.setp(ax.get_xticklabels(), rotation=15, ha='right'); show()


## Recall alto explica F1 alto?


In [ ]:
preds = ctx.all_predictions()
if preds.empty or 'recall.recall_context' not in preds.columns:
    print('sem predictions.jsonl com recall')
else:
    b = preds.copy()
    b['bucket'] = pd.cut(b['recall.recall_context'], [-.01, .001, .34, .67, 1.0],
                         labels=['0', '0–⅓', '⅓–⅔', '⅔–1'])
    piv = b.pivot_table(index='bucket', columns='condition', values='f1', observed=True)
    ax = piv.plot(kind='bar', figsize=(9, 3.6), color=colors(piv.columns), width=.8)
    ax.set_xlabel('recall do contexto'); ax.set_ylabel('F1 médio')
    ax.set_title('F1 condicionado ao recall da evidência')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8); show()
    display(piv.round(3))


## Custo: ingestão vs QA


In [ ]:
plot_cost(ctx); show()


In [ ]:
ctx.cost.round(1)


## Custo por propósito de chamada

Mostra exatamente para onde vai o orçamento: extração, resolução de
entidades, sigma-agent, curadoria, ou geração de respostas.


In [ ]:
rows = []
for cond in ctx.conditions:
    for purpose, v in ctx.raw[cond].get('cost', {}).get('by_purpose', {}).items():
        rows.append({'condition': cond, 'purpose': purpose,
                     'calls': v.get('calls', 0),
                     'tokens': v.get('prompt_tokens', 0) + v.get('completion_tokens', 0)})
by_purpose = pd.DataFrame(rows)
if by_purpose.empty:
    print('sem detalhamento por propósito')
else:
    piv = by_purpose.pivot_table(index='condition', columns='purpose',
                                 values='tokens', aggfunc='sum').fillna(0)
    ax = piv.plot(kind='barh', stacked=True, figsize=(10, 3.8))
    ax.set_xlabel('tokens'); ax.set_ylabel('')
    ax.set_title('Tokens por propósito de chamada')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8); show()
    display(piv.astype(int))


## Eficiência: F1 por 1k tokens

B1 (contexto completo) costuma pagar caro por cada ponto de F1; é aí que
uma memória compacta precisa justificar sua existência.


In [ ]:
eff = ctx.cost.join(ctx.overall[['f1_micro']])
eff['f1_per_1k_tokens'] = eff.f1_micro / (eff.tokens_total / 1000).replace(0, float('nan'))
display(eff[['f1_micro', 'tokens_total', 'mean_context_tokens', 'f1_per_1k_tokens']].round(4))
ax = eff['f1_per_1k_tokens'].plot(kind='barh', figsize=(8, 3.2),
                                  color=colors(eff.index))
ax.set_xlabel('F1 por 1k tokens gastos'); ax.set_ylabel(''); show()
